# 大きなモデルの予測確率で小さなモデルを学習する（知識蒸留）

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/12_classification/knowledge_distillation.ipynb)

NLP クックブック 12-22 ／ テキスト分類

> GPU ランタイム推奨（ランタイム → ランタイムのタイプを変更 → T4 GPU）

ファインチューニングした大きな BERT は精度が高いが、CPU で大量の文書をさばくには重い。小さなモデル（生徒）を学習するときに、大きな BERT（教師）が出す各クラスの確率（ソフトラベル）も手本にして（知識蒸留）、生徒の精度を上げたい。livedoor ニュースコーパスの記事の見出しだけから 9 カテゴリを当てる課題で、生徒を普通に学習した場合と比べ、パラメータ数と推論の速さ、温度と混合の重みの影響も調べる。

In [1]:
!pip install -q fugashi unidic-lite   # 教師（東北大 BERT）のトークナイザーが MeCab を使う

In [2]:
import io, tarfile, urllib.request
import pandas as pd
from datasets import ClassLabel, Dataset

def load_livedoor():
    """livedoor ニュースコーパス（株式会社ロンウイット配布、記事は CC BY-ND 2.1 JP）を DataFrame にする"""
    url = "https://www.rondhuit.com/download/ldcc-20140209.tar.gz"
    with urllib.request.urlopen(url) as r:
        tar = tarfile.open(fileobj=io.BytesIO(r.read()), mode="r:gz")
    rows = []
    for m in tar.getmembers():
        parts = m.name.split("/")
        if m.isfile() and len(parts) == 3 and parts[2] != "LICENSE.txt":
            lines = tar.extractfile(m).read().decode("utf-8").split("\n")   # 1 行目 URL, 2 行目 日付, 3 行目 タイトル
            rows.append({"label": parts[1], "title": lines[2], "text": "\n".join(lines[3:])})
    return pd.DataFrame(rows)

df = load_livedoor()
names = sorted(df["label"].unique())
ds = Dataset.from_pandas(df).cast_column("label", ClassLabel(names=names))
ds = ds.train_test_split(test_size=0.2, stratify_by_column="label", seed=0)   # この章の「BERT をファインチューニングして文書分類する (Trainer)」と同じ分割
print(ds)

DatasetDict({
    train: Dataset({
        features: ['label', 'title', 'text'],
        num_rows: 5893
    })
    test: Dataset({
        features: ['label', 'title', 'text'],
        num_rows: 1474
    })
})


In [3]:
import time
import numpy as np
import torch
import torch.nn.functional as F
from transformers import (AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding,
                          PrinterCallback, Trainer, TrainingArguments, set_seed)
from transformers.utils import logging

logging.set_verbosity_error()   # 分類用の層を新しく初期化したという報告を出さない
BF16 = torch.cuda.get_device_capability()[0] >= 8   # bfloat16 は Ampere (A100, L4) 以降。T4 では float16 を使う
MAX_LEN = 64   # 見出しだけを入力にする

TEACHER = "tohoku-nlp/bert-large-japanese-v2"  # 東北大 BERT-large（24 層、隠れ層 1024 次元、Apache 2.0）
STUDENT = "sbintuitions/modernbert-ja-30m"     # SB Intuitions の ModernBERT-Ja（10 層、隠れ層 256 次元、MIT）

def encode(name):
    tok = AutoTokenizer.from_pretrained(name)
    enc = ds.map(lambda b: tok(b["title"], truncation=True, max_length=MAX_LEN,
                               return_token_type_ids=False),   # ModernBERT は token_type_ids を受け取らない
                 batched=True, remove_columns=["title", "text"])
    return tok, enc

class DistillTrainer(Trainer):
    """損失 = alpha × (温度 T の KL ダイバージェンス × T²) + (1 − alpha) × 正解ラベルとの交差エントロピー"""
    def __init__(self, *args, T=2.0, alpha=0.5, **kwargs):
        super().__init__(*args, **kwargs)
        self.T, self.alpha = T, alpha

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels, t_logits = inputs.pop("labels"), inputs.pop("teacher_logits", None)   # 評価のときは教師の出力がない
        outputs = model(**inputs)
        s_logits = outputs.logits.float()
        loss = F.cross_entropy(s_logits, labels)
        if t_logits is not None and self.alpha > 0:
            T = self.T
            kd = F.kl_div(F.log_softmax(s_logits / T, dim=-1), F.log_softmax(t_logits.float() / T, dim=-1),
                          reduction="batchmean", log_target=True) * T * T   # T² を掛けて勾配の大きさを温度によらずそろえる
            loss = self.alpha * kd + (1 - self.alpha) * loss
        return (loss, outputs) if return_outputs else loss

def train(name, tok, train_ds, epochs, lr, T=2.0, alpha=0.0, seed=0):
    set_seed(seed)
    model = AutoModelForSequenceClassification.from_pretrained(
        name, num_labels=len(names), id2label=dict(enumerate(names)), label2id={n: i for i, n in enumerate(names)})
    pad = DataCollatorWithPadding(tok)
    def collate(features):   # 教師の出力（9 次元）は長さがそろっているので、そのまま積む
        t = [f.pop("teacher_logits") for f in features] if "teacher_logits" in features[0] else None
        batch = pad(features)
        if t is not None:
            batch["teacher_logits"] = torch.tensor(t)
        return batch
    args = TrainingArguments(output_dir="out", num_train_epochs=epochs, learning_rate=lr, warmup_steps=0.1,   # 1 未満は全ステップ数に対する割合
                             weight_decay=0.01, per_device_train_batch_size=32, per_device_eval_batch_size=128, save_strategy="no",
                             logging_strategy="no", report_to="none", bf16=BF16, fp16=not BF16, seed=seed, disable_tqdm=True,
                             remove_unused_columns=False)   # teacher_logits の列を消させない
    trainer = DistillTrainer(model=model, args=args, train_dataset=train_ds, data_collator=collate, T=T, alpha=alpha)
    trainer.remove_callback(PrinterCallback)   # 学習のたびに最後の集計（train_runtime など）を表示させない
    trainer.train()
    return trainer

def predict(trainer, test_ds):
    return trainer.predict(test_ds).predictions

In [4]:
# 教師: 東北大 BERT-large をファインチューニングする（Colab の T4 で 5〜7 分の見込み）
t0 = time.time()
tok_t, enc_t = encode(TEACHER)
teacher = train(TEACHER, tok_t, enc_t["train"], epochs=3, lr=2e-5)
gold = np.array(enc_t["test"]["label"])
logits_t = predict(teacher, enc_t["test"])
print(f"教師の正解率: {(logits_t.argmax(-1) == gold).mean():.3f}（{time.time() - t0:.0f} 秒）")

# 学習データに対する教師の出力（ロジット）を一度だけ計算して保存しておく（オフラインの蒸留）
train_logits = predict(teacher, enc_t["train"])
p = torch.tensor(train_logits).float()
for T in [1, 2, 4]:
    q = F.softmax(p / T, dim=-1)
    print(f"T={T}: 学習データでの教師の最大確率の平均 {q.max(-1).values.mean():.3f}、"
          f"2 番目に高い確率の平均 {q.topk(2).values[:, 1].mean():.4f}")

教師の正解率: 0.927（58 秒）
T=1: 学習データでの教師の最大確率の平均 0.984、2 番目に高い確率の平均 0.0095
T=2: 学習データでの教師の最大確率の平均 0.837、2 番目に高い確率の平均 0.0440
T=4: 学習データでの教師の最大確率の平均 0.462、2 番目に高い確率の平均 0.0959


In [5]:
# 生徒: ModernBERT-Ja-30M を、正解ラベルだけで学習した場合 (alpha=0) と蒸留した場合で比べる。
# 乱数の種を 3 つ変えて学習する（Colab の T4 で 1 回 1 分前後、合計 5〜7 分の見込み）
tok_s, enc_s = encode(STUDENT)
train_s = enc_s["train"].add_column("teacher_logits", train_logits.tolist())   # ds の同じ並びなので行がそろっている
PRED = {}
def run(T, alpha, seed=0):
    tr = train(STUDENT, tok_s, train_s, epochs=3, lr=1e-4, T=T, alpha=alpha, seed=seed)
    PRED[(T, alpha, seed)] = predict(tr, enc_s["test"]).argmax(-1)
    return tr

for seed in [0, 1, 2]:
    student_base = run(T=1, alpha=0.0, seed=seed)   # 普通の学習（交差エントロピーだけ）
    student_kd = run(T=2, alpha=0.5, seed=seed)     # 蒸留
for T, alpha, label in [(1, 0.0, "普通の学習"), (2, 0.5, "蒸留 (T=2, alpha=0.5)")]:
    accs = [(PRED[(T, alpha, s)] == gold).mean() for s in [0, 1, 2]]
    print(f"{label}: 正解率 {np.mean(accs):.3f}（種ごと {', '.join(f'{a:.3f}' for a in accs)}）")

普通の学習: 正解率 0.897（種ごと 0.898, 0.898, 0.896）
蒸留 (T=2, alpha=0.5): 正解率 0.909（種ごと 0.906, 0.909, 0.913）


In [6]:
# 温度と混合の重みを変える（種は 0 だけ。Colab の T4 で 4〜6 分の見込み）
from sklearn.metrics import f1_score

for T, alpha in [(1, 0.5), (4, 0.5), (8, 0.5), (2, 0.9), (2, 1.0)]:
    run(T, alpha)
table = pd.DataFrame([{"T": T, "alpha": a, "正解率": (p == gold).mean(), "macro-F1": f1_score(gold, p, average="macro")}
                      for (T, a, s), p in PRED.items() if s == 0]).sort_values(["alpha", "T"])
print(table.round(3).to_string(index=False))

 T  alpha   正解率  macro-F1
 1    0.0 0.898     0.892
 1    0.5 0.903     0.897
 2    0.5 0.906     0.900
 4    0.5 0.904     0.899
 8    0.5 0.904     0.900
 2    0.9 0.908     0.903
 2    1.0 0.915     0.910


In [7]:
# テストの 1474 件をブートストラップして、差の 95% 信頼区間を出す（生徒は 3 つの種の正解率の平均。評価データの抽出によるばらつきだけを見る）
c_base, c_kd = [np.mean([PRED[(T, a, s)] == gold for s in [0, 1, 2]], axis=0) for T, a in [(1, 0.0), (2, 0.5)]]
c_t = (logits_t.argmax(-1) == gold).astype(float)
rng = np.random.default_rng(0)
idx = rng.integers(0, len(gold), (1000, len(gold)))   # 同じ組を引き直す（対応のあるブートストラップ）
for name, a, b in [("蒸留 − 普通の学習", c_kd, c_base), ("教師 − 蒸留した生徒", c_t, c_kd)]:
    lo, hi = np.percentile((a[idx] - b[idx]).mean(1), [2.5, 97.5])
    print(f"{name}: {a.mean() - b.mean():+.3f}（95% 信頼区間 {lo:+.3f}〜{hi:+.3f}）")

蒸留 − 普通の学習: +0.012（95% 信頼区間 +0.005〜+0.019）
教師 − 蒸留した生徒: +0.018（95% 信頼区間 +0.007〜+0.030）


In [8]:
# パラメータ数と推論の速さ（トークナイズの時間は含めない。Colab で 3〜4 分の見込み）
import contextlib

def count(model):
    total = sum(p.numel() for p in model.parameters())
    emb = model.get_input_embeddings().weight.numel()
    return total / 1e6, (total - emb) / 1e6

def bench(model, tok, device, n, bs):
    sub = ds["test"].select(range(n))
    batches = [tok(sub["title"][i:i + bs], truncation=True, max_length=MAX_LEN,
                   padding=True, return_token_type_ids=False, return_tensors="pt") for i in range(0, n, bs)]
    model = model.to(device).eval()
    if device == "cuda":
        ctx = torch.autocast("cuda", dtype=torch.bfloat16 if BF16 else torch.float16)
    else:
        ctx = contextlib.nullcontext()   # CPU は float32 のまま
    with torch.inference_mode(), ctx:
        model(**{k: v.to(device) for k, v in batches[0].items()})   # 1 回目は準備に時間がかかるので測らない
        best = float("inf")
        for _ in range(3):   # 3 回測って最も速い回をとる（ほかの処理の影響を減らす）
            if device == "cuda": torch.cuda.synchronize()
            t0 = time.perf_counter()
            for b in batches:
                model(**{k: v.to(device) for k, v in b.items()})
            if device == "cuda": torch.cuda.synchronize()
            best = min(best, time.perf_counter() - t0)
    return n / best

print("CPU のスレッド数:", torch.get_num_threads())
for name, tr, tok in [("教師 (BERT-large)", teacher, tok_t), ("生徒 (ModernBERT-Ja-30M)", student_kd, tok_s)]:
    total, body = count(tr.model)
    gpu = bench(tr.model, tok, "cuda", n=1024, bs=64)
    cpu = bench(tr.model, tok, "cpu", n=128, bs=32)
    print(f"{name}: パラメータ {total:.0f}M（埋め込みを除くと {body:.0f}M）、GPU {gpu:.0f} 件/秒、CPU {cpu:.1f} 件/秒")
print(f"GPU メモリの最大使用量: {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")

CPU のスレッド数: 2
教師 (BERT-large): パラメータ 337M（埋め込みを除くと 304M）、GPU 1577 件/秒、CPU 5.7 件/秒
生徒 (ModernBERT-Ja-30M): パラメータ 37M（埋め込みを除くと 11M）、GPU 9746 件/秒、CPU 170.8 件/秒
GPU メモリの最大使用量: 6.9 GB


In [9]:
# 蒸留した生徒（種 2）を保存して、pipeline で CPU で推論する
from transformers import pipeline

student_kd.save_model("livedoor-student"); tok_s.save_pretrained("livedoor-student")
clf = pipeline("text-classification", model="livedoor-student", device="cpu")
for title in ["新型スマートフォン、有機 EL と高性能カメラを搭載して来月発売へ", "日本代表が後半に逆転、W 杯予選で勝利", "30 代独身女性が語る、結婚に踏み切れない本当の理由"]:
    print(clf(title), title)

[{'label': 'smax', 'score': 0.6795187592506409}] 新型スマートフォン、有機 EL と高性能カメラを搭載して来月発売へ
[{'label': 'sports-watch', 'score': 0.9929988384246826}] 日本代表が後半に逆転、W 杯予選で勝利
[{'label': 'dokujo-tsushin', 'score': 0.9587757587432861}] 30 代独身女性が語る、結婚に踏み切れない本当の理由


- 知識蒸留（Hinton ら 2015）では、生徒に正解ラベルだけでなく、教師の確率の分布（「家電チャンネル」らしいが「IT ライフハック」の可能性も少しある、など）を真似させる。温度 T で割ってからソフトマックスをとると分布がなだらかになり、2 番目以下のクラスの情報が損失に効くようになる。T² を掛けるのは、温度を変えても蒸留の項の勾配の大きさをそろえるため（同論文）。損失の関数は短いので、Trainer の compute_loss を上書きして自分で書けばよい。教師の学習はこの章の「BERT をファインチューニングして文書分類する (Trainer)」とほぼ同じ手順だが、入力を見出しだけにし、モデルを BERT-large に変え、最良のエポックを選ぶ処理は省いた。なお、`test_size=0.2, seed=0` で得られるテストセット（1,474 件）は「BERT をファインチューニングして文書分類する (Trainer)」の `test` と完全に同一だが、このレシピでは開発セット (`dev`) を切り分けず残りの 80% 全体を学習に使っている。
- 教師は自分が学習したデータに対しては自信が強く、T=1 では最大確率の平均が 0.984、2 番目のクラスの確率の平均は 0.0095 でほぼ正解ラベルと同じだった。T=2 では 0.837 と 0.044、T=4 では 0.462 と 0.096 になり、2 番目以下のクラスの情報が表に出る。ラベルのない文書が別にあれば、教師に確率を付けさせて生徒の学習データに足すこともできる。
- 見出しだけの課題では、教師 (BERT-large) の正解率は 0.927、生徒を普通に学習すると 0.897（3 つの種で 0.896〜0.898）、蒸留 (T=2, alpha=0.5) すると 0.909 (0.906〜0.913) で、3 つの種のすべてで蒸留のほうが高かった。差は +0.012（95% 信頼区間 +0.005〜+0.019）で、教師との差 0.030 の 4 割ほどを埋めたが、教師とはまだ 0.018（同 +0.007〜+0.030）の差がある。区間の求め方は評価指標と統計的検定の章の「ブートストラップ法で評価値の信頼区間を求める」を参照。
- 温度と混合の重みを変えた表（種 0 のみ）では、蒸留したどの設定 (0.903〜0.915) も普通の学習 (0.898) より高く、教師の確率だけで学習した alpha=1.0 が最も高かった (0.915)。ただし設定どうしの差は 1.2 ポイント程度で、1 つの種、1474 件の評価では見分けにくい。また、ここで `(T, alpha)` の組み合わせをテストデータで比べているのは感度分析（アブレーション実験）のためであり、実際にモデルやハイパーパラメータを選ぶときは学習データから検証用のデータを分けて温度と重みを決める。
- 生徒のパラメータは 37M で教師 (337M) の約 9 分の 1、埋め込みを除くと 11M と 304M で約 28 分の 1 である（ModernBERT-Ja は語彙が 102,400 語と大きい）。2 スレッドの CPU では、教師が 5.7 件/秒、生徒が 170.8 件/秒で、生徒が約 30 倍速かった。GPU (float16) でも生徒が約 6 倍速かったが、出力の GPU の件/秒の値は検証に使った GPU（T4 より速い）で測ったもので、Colab の T4 ではどちらも小さくなる。速さは環境や負荷で変わるので、Colab では測り直す。GPU メモリの最大使用量は 6.9 GB だった。
- ModernBERT-Ja-30M のモデルカードにある livedoor ニュースコーパスの評価（入力や分割はこのレシピと異なる）では、正解率が ModernBERT-Ja-30M で 97.20、東北大 BERT-base v3 で 96.91、BERT-large v2 で 97.64 で、モデルの大きさによる差がほとんどない。このような課題では蒸留の余地は小さい。このレシピで見出しだけを入力にしたのは、教師と生徒の差が出る課題にするためである。教師と生徒は語彙もトークナイザー（MeCab と WordPiece、SentencePiece）も違うが、出力の確率だけを使うので問題にならない。
- データの livedoor ニュースコーパスは CC BY-ND 2.1 JP（改変禁止）である。教師は東北大の BERT-large (Apache 2.0)、生徒は SB Intuitions の ModernBERT-Ja-30M (MIT)。学習したモデルを配布するときは、データのライセンスとの関係に注意する。